# OOV · Maltese ASR
**Colab:** Runtime → Change runtime type → **T4 GPU**. (Kaggle: Accelerator GPU T4, Internet on.)
Run top to bottom; download `results.zip` at the end.

In [ ]:
!rm -rf oov-maltese-asr && git clone -q https://github.com/exhilarin/oov-maltese-asr.git
%cd oov-maltese-asr
!pip install -q jiwer
!python src/common.py && python src/split_stats.py
# smoke test: 8 dev clips. If this prints an error, stop here and send the error.
!python src/transcribe.py openai/whisper-small --split dev --limit 8

In [ ]:
# HF baselines on test (one pass each)
!python src/transcribe.py carlosdanielhernandezmena/wav2vec2-large-xlsr-53-maltese-64h
!python src/transcribe.py openai/whisper-small

In [ ]:
# Our model: fine-tune whisper-small (best ckpt by dev WER), then test ONCE
!python src/finetune_whisper.py --steps 1000 --out whisper-small-mt
!python src/transcribe.py whisper-small-mt --split dev
!python src/transcribe.py whisper-small-mt

In [ ]:
# NeMo baselines AFTER fine-tune: the heavy install may change library versions
!pip install -q "nemo_toolkit[asr]"
!python src/transcribe.py nvidia/parakeet-tdt-0.6b-v3
!python src/transcribe.py nvidia/canary-1b-v2 --batch 8

In [ ]:
!python src/score.py --split test
!python src/error_analysis.py --split test
!zip -qr results.zip results